# 👥 Sales Team Performance & Retention Risk Analysis
### Author: Sarthak Agarwal
**Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, PostgreSQL, Power BI

---

## 📌 Executive Summary
This project analyzes a corporate workforce dataset of **1,470 employee records** to evaluate retention risks, diagnose workload burnout drivers, and evaluate productivity across commercial teams.

### Core Project Objectives (Aligned with Resume):
1. **Workforce Baseline Analytics:** Evaluate 1,470 employee records in Python, establishing an empirical **16.12% baseline attrition rate**.
2. **Commercial Team Deep Dive:** Uncover the acute **39.76% turnover rate** concentrated among **Sales Representatives**.
3. **Predictive Risk Scoring:** Segment workforce profiles by tenure, workload (overtime), and satisfaction to generate proactive intervention alerts.
4. **Team Incentive Restructuring:** Formulate data-backed recommendations for sales quota pacing, commission structures, and early-career retention.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('Analytics environment loaded successfully.')


## 1. 📂 Data Ingestion & Data Hygiene


In [ ]:
df = pd.read_csv('../data/HR_Analytics.csv')
print(f'Dataset Dimensions: {df.shape[0]:,} rows and {df.shape[1]} features')
df.head(3)


In [ ]:
# Normalize column names to snake_case
df.columns = df.columns.str.replace(r'(?<!^)(?=[A-Z])', '_', regex=True).str.lower()

# Remove non-informative constants
drop_cols = ['employee_count', 'employee_number', 'over18', 'standard_hours']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Binary attrition flag for calculations
df['attrition_flag'] = (df['attrition'] == 'Yes').astype(int)

print(f'Remaining clean features: {df.shape[1]}')
df.info()


## 2. 📊 Baseline Attrition & Departmental Benchmarking


In [ ]:
total_employees = len(df)
departed_employees = df['attrition_flag'].sum()
baseline_rate = (departed_employees / total_employees) * 100

print(f'Total Workforce: {total_employees:,}')
print(f'Total Attrition: {departed_employees:,} employees')
print(f'Baseline Company-Wide Attrition Rate: {baseline_rate:.2f}%')


In [ ]:
dept_summary = df.groupby('department').agg(
    headcount=('age', 'count'),
    departures=('attrition_flag', 'sum'),
    attrition_rate=('attrition_flag', lambda x: round(x.mean() * 100, 2)),
    avg_income=('monthly_income', lambda x: round(x.mean(), 2)),
    avg_satisfaction=('job_satisfaction', lambda x: round(x.mean(), 2))
).reset_index().sort_values(by='attrition_rate', ascending=False)

dept_summary


## 3. 🎯 Commercial Role Analysis: The 39.76% Sales Representative Turnover Discovery


In [ ]:
role_summary = df.groupby(['department', 'job_role']).agg(
    headcount=('age', 'count'),
    departures=('attrition_flag', 'sum'),
    attrition_rate=('attrition_flag', lambda x: round(x.mean() * 100, 2)),
    avg_income=('monthly_income', lambda x: round(x.mean(), 2))
).reset_index().sort_values(by='attrition_rate', ascending=False)

role_summary


In [ ]:
plt.figure(figsize=(11, 5))
role_plot = df.groupby('job_role')['attrition_flag'].mean().sort_values(ascending=False) * 100
colors = ['#d95f02' if 'Sales Representative' in r else '#2b5c8f' for r in role_plot.index]

bars = plt.barh(role_plot.index, role_plot.values, color=colors, edgecolor='black', alpha=0.9)
plt.axvline(baseline_rate, color='red', linestyle='--', label=f'Baseline Attrition ({baseline_rate:.2f}%)')
plt.title('Attrition Rate across Job Roles (Sales Representatives: 39.76%)', fontsize=12, fontweight='bold')
plt.xlabel('Attrition Rate (%)', fontsize=10)
plt.gca().invert_yaxis()
plt.legend(frameon=True)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.8, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()


## 4. ⏱️ Workload Fatigue & Overtime Burnout Gap
Evaluating the burnout turnover gap between employees working regular hours vs overtime.


In [ ]:
ot_summary = df.groupby('over_time').agg(
    headcount=('age', 'count'),
    departures=('attrition_flag', 'sum'),
    attrition_rate=('attrition_flag', lambda x: round(x.mean() * 100, 2)),
    avg_income=('monthly_income', lambda x: round(x.mean(), 2)),
    avg_wlb=('work_life_balance', lambda x: round(x.mean(), 2))
).reset_index()

ot_summary


## 5. ⏳ Tenure Cohorts & Early-Stage Retention Risk
Analyzing employee churn by tenure bands (0-1 years, 2-5 years, etc.).


In [ ]:
df['tenure_band'] = pd.cut(df['years_at_company'], bins=[-1, 1, 5, 10, 45], labels=['0-1 Years', '2-5 Years', '6-10 Years', '10+ Years'])

tenure_summary = df.groupby('tenure_band', observed=False).agg(
    headcount=('age', 'count'),
    departures=('attrition_flag', 'sum'),
    attrition_rate=('attrition_flag', lambda x: round(x.mean() * 100, 2))
).reset_index()

tenure_summary


## 6. 🚨 Predictive Risk Scoring Framework
We engineer a multi-factor risk scoring model based on:
1. Workload (Overtime required)
2. Commute friction (Far distance)
3. Promotion stagnation (5+ years without promotion)
4. Work-Life balance dissatisfaction (Bad / Average)
5. Low job satisfaction (Low / Moderate)
6. Early tenure vulnerability (0-1 years tenure)


In [ ]:
# Feature engineered bands
df['distance_band'] = pd.cut(df['distance_from_home'], bins=[-1, 5, 15, 50], labels=['Near', 'Moderate', 'Far'])
df['promotion_gap_band'] = pd.cut(df['years_since_last_promotion'], bins=[-1, 1, 4, 20], labels=['0-1 Years', '2-4 Years', '5+ Years'])
df['job_satisfaction_band'] = df['job_satisfaction'].map({1: 'Low Satisfaction', 2: 'Moderate Satisfaction', 3: 'High Satisfaction', 4: 'Very High Satisfaction'})
df['work_life_balance_band'] = df['work_life_balance'].map({1: 'Bad', 2: 'Average', 3: 'Good', 4: 'Best'})

# Multi-Factor Score Calculation
df['risk_score'] = (
    (df['over_time'] == 'Yes').astype(int) +
    (df['distance_band'] == 'Far').astype(int) +
    (df['promotion_gap_band'] == '5+ Years').astype(int) +
    (df['work_life_balance_band'].isin(['Bad', 'Average'])).astype(int) +
    (df['job_satisfaction_band'].isin(['Low Satisfaction', 'Moderate Satisfaction'])).astype(int) +
    (df['tenure_band'] == '0-1 Years').astype(int)
)

df['risk_level'] = df['risk_score'].apply(lambda x: 'Low Risk' if x <= 2 else ('Medium Risk' if x <= 3 else 'High Risk'))

risk_perf = df.groupby('risk_level', observed=False).agg(
    headcount=('age', 'count'),
    departures=('attrition_flag', 'sum'),
    actual_attrition_rate=('attrition_flag', lambda x: round(x.mean() * 100, 2))
).reset_index()

risk_perf


In [ ]:
# Flag active Sales accounts requiring immediate retention intervention
sales_risk_pipeline = df[(df['department'] == 'Sales') & (df['attrition'] == 'No') & (df['risk_level'] == 'High Risk')].copy()
sales_risk_pipeline.sort_values(by=['risk_score', 'monthly_income'], ascending=[False, True], inplace=True)

print(f'Active High-Risk Sales Employees: {len(sales_risk_pipeline):,} accounts')
sales_risk_pipeline[['job_role', 'monthly_income', 'over_time', 'job_satisfaction', 'years_at_company', 'risk_score']].head(10)


## 7. 💡 Strategic Business Recommendations & Incentive Restructuring

### A. Sales Representative Compensation & Incentive Restructuring
- **Root Cause of 39.76% Turnover:** Sales Representatives receive an average monthly income of only **,626** (lowest corporate salary tier) combined with intense quota pressure.
- **Actionable Strategy:** Introduce quarterly milestone bonuses, tiered commission accelerators for early-stage reps, and guaranteed base salary adjustments after 6 months of quota achievement.

### B. Overtime & Workload Distribution
- **The Burnout Factor:** Overtime employees suffer a **30.53% turnover rate**, nearly 3x higher than non-overtime peers (10.44%).
- **Actionable Strategy:** Cap mandatory overtime for commercial teams, institute compensatory time-off (comp days), and redistribute inbound SDR leads evenly.

### C. First-Year Retention Program
- **Early Churn Hotspot:** Employees in their first year experience **34.88% attrition**.
- **Actionable Strategy:** Implement structured 30-60-90 day onboarding check-ins and formal peer mentoring programs to reduce early drop-off.


In [ ]:
# Export final sanitized dataset for SQL and Power BI ingestion
df.to_csv('../data/cleaned_employee_attrition.csv', index=False)
print('Dataset saved to ../data/cleaned_employee_attrition.csv')
